# <center>Organizaci&oacute;n de Datos</center>
#### <center>C&aacute;tedra Ing. Rodriguez, Juan Manuel </center>

## <center>Procesamiento de Lenguaje Natural (NLP)</center>
### <center> Redes Neuronales Recurrentes </center>


# Referencias:

- [Curso NLP de Instituto Humai. Clase 2, Redes Recurrentes](https://github.com/institutohumai/cursos-python/tree/master/NLP/2_Redes_Recurrentes)


# Modelos de secuencias

Hasta ahora hemos trabajado con series de datos donde a cada entrada le corresponde una salida. Por ejemplo, a una imagen le corresponde una categoría. A una serie de indicadores biométricos le corresponde un diagnóstico médico.

En procesamiento de lenguajes naturales, nuestras salidas y nuestras entradas tienen una característica distinta. Veamos un ejemplo:

> *Usted tiene 16 años. Está prohibido vender alcohol a menores de 18 años. No puedo venderle esa botella.*

En el ejemplo anterior, tenemos tres afirmaciones, donde la última es un conclusión de las dos anteriores. En este sentido, cuando trabajamos con lenguajes tenemos el problema de lo próximo que se dice, depende de lo que se dijo antes. Es decir, estamos trabajando con secuencias temporales.

Peor aún, muchas veces la última salida, debe ser tratada como una nueva entrada. Piense en el ejemplo anterior, si usted vive en un país latinoamericano o europeo, al llegar a **menores de** intuía que la edad límite sería **18 años**. Eso es porque como ciudadano de su país, sabe que esa es la ley. Es decir. **18 años** podría haber sido una predicción, una salida de su red. Ademas, al haber predicho **18 años** ahora podemos concluir que **No puedo venderle esa botella**. Si la ley dijera que los menores de **14 años** pueden comprar alcohol, la segunda frase carecería de sentido. Es decir **18 años** es una predicción, una salida en un momento, pero luego se convierte en una entrada o un *feature* en otro.

Es por lo anterior que se dice que estos modelos son modelos autoregresivos: las salidas luego se convierten en entradas, como en un problema recursivo.

La naturaleza autogresiva de nuestro modelo hace que debamos considerar la calidad de nuestras predicciones. Volviendo al ejemplo anterior, si nuestra predicción hubiera sido **14 años** en lugar de **18 años**, la conclusión final de nuestra frase sería distinta a la que hemos obtenido. Pequeños errores en un nuestras predicciones pueden acumularse a lo largo del tiempo y generar resultados absurdos.



# Variables ocultas.

Volvamos de nuevo a nuestro ejemplo de oraciones

> *Usted tiene 16 años. Está prohibido vender alcohol a menores de 18 años. No puedo venderle esa botella.*

Supongamos de nuevo que queremos predecir **18 años**. La cantidad de palabras escritas hasta ese momento es 11. Luego de predecir **18 años**, la cantidad de palabras aumentó a 13. Es decir, conforme predecimos y agregamos información, nuestro modelo debe responder a la cantidad creciente de palabras o ejemplos.

Recordemos que todas estas herramientas nacieron de la estadística, por lo que nuestras predicciones se basaran en considerar la probabilidad de que diferentes palabras ocurran en simultaneo. Esto es verdaderamente un problema: mientas más palabras tenemos, menos probable es que vuelvan a ocurrir. Si ocurren infrecuentemente, necesitamos aumentar cada vez más la cantidad de ejemplos de nuestros datos. Esto puede ser un problema incluso para oraciones cortas. Una alternativa para paliar este problema es limitar la cantidad de palabra que miraremos hacia atras.

Otra alternativa a esto es trabajar con **variables ocultas**. Las variables ocultas son cantidades que de alguna manera agrupan la información de todos los casos anteriores. Por ejemplo:

> *Usted es menor de edad. No puedo venderle esa botella.*

Hemos resumido toda la información de dos oraciones en una sola mucho más corta.

De la misma manera que buscamos representaciones abstractas para palabras por medio de *tokens*, usaremos esos tokens para generar nuevas variables que resuman la información anterior. Es decir, generaremos una variable que de alguna manera tiene toda la información de **Usted es menor de edad**

Al trabajar con variables ocultas, esperamos reemplazar las todas las palabras anteriores con el último valor de la variable oculta. Así, nuestro problema que antes veía 13 variables o tokens ahora ve uno solo. Esto nos permite simplificar nuestro modelo para trabajar con **modelos makovianos de primer orden** (sin entrar en mucho detalle, los modelos markovianos son un tipo de modelos autoregresivos).



## Esbozo de la noción de modelo de Markov

* Tenemos un estado (las últimas palabras escritas) a la cual llegamos a partir de un estado inicial bien definido
* Tenemos una historia de estados pasados que afecta  a estados futuros (cada palabra predicha o dentro de nuestro dataset)
* Hay una probabilidad asosciada a cada cambio de estado
* Queremos predecir cual será el próximo estado de un grupo finito de estados (la próxima palabra).

Al trabajar con un modelo markoviano sobre variables ocultas, esperamos que la variable oculta resuma con tanta fidelidad los tokens pasados que solo necesitemos la variable oculta más reciente. Al necesitar solo la más reciente, se dice que es un modelo markoviano de primer orden (requiera solo una variable anterior). La razón por la que buscamos trabajar con modelos de primer orden es que son menos costosos computacionalmente.

En resumen nuestra propuesta para generar modelos de lenguaje consistira en lo siguiente:

1. Tomaremos texto de para crear nuestro *datset*
1. Transformaremos nuestro texto en algun tipo de representación simbólica (*tokens*)
2. De esta manera, nuestro modelo de lenguaje se convertirá en un problema de clasificiación: Dadas las palabras anteriores, ¿Cuál es la siguiente palabra?
  * Decimos que es un problema de clasificación, porque cada una de nuestra palabras es una categoría.
3. Para crear nuestro modelo de lenguaje, usaremos variables ocultas en el contexto de un modelo markoviano.
  * La justificación para esto es que el lenguaje tiene características de un modelo markoviano.



# Redes neuronales recurrentes

En la sección anterior intentamos argumentar que el lenguaje puede modelarse con un modelo markoviano. Además, propusimos trabajar con modelos markovinos con variables ocultas. La idea de trabajar con variables ocultas era poder trabajar con un modelo markoviano de primer orden. Queremos usar estos modelos de primer orden porque sabemos que nos permitiran ahorrar uso de memoria, así como disminuir el uso de recursos computacionales.

Nuestra propuesta para trabajar con variables ocultas, será trabajar con las unidades ocultas de un perceptrón multicapa

![](http://d2l.ai/_images/mlp.svg)

$$\mathbf{O} = \mathbf{H} \mathbf{W} + \mathbf{b}$$
$$\mathbf{H} = \phi(\mathbf{X} \mathbf{W} + \mathbf{b})$$

Sin embargo, como trabajaremos con secuencias temporales, nuestra entrada al tiempo $t$, debe depender del tiempo $t-1$. Es decir, la próxima palabra debe depender de las palabras anteriores. En un perceptrón multicapa, esa dependecia temporal no está presente. Es por esto que debemos reestructurar nuestra capa para que permita generar modelos autoregresivos de secuencias. Dado queremos usar las variables ocultas como cantidades que resumen toda la información anterior, son estas cantidades las que tendran una dependencia temporal

$$\mathbf{O}_{t} = \mathbf{H}_{t} \mathbf{W}_{O} + \mathbf{b}$$
$$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_{X} + \mathbf{H}_{t-1} \mathbf{W}_{H}  + \mathbf{b}_h).$$

Notemos que $\mathbf{H}_t$ depende del valor anterior, $\mathbf{H}_{t-1}$ y del nuevo valor $\mathbf{X}_t$. Esta dependencia temporal es la que hace que nuestra nueva red neuronal sea una *red neuronal recurrente*. Recordemos si nuestro modelo está correctamente entrenado la salida $\mathbf{O}_t$ debe coincidir con el resultado correcto o *grounding truth* de $\mathbf{X}_{t+1}$. Esta era la naturaleza autoregresiva de nuestros modelos.

En la siguiente figura mostramos el proceso de cálculo nuestra capa recurrente

![An RNN with a hidden state.](http://d2l.ai/_images/rnn.svg)

En la figura, vemos que ocurre a cada instante $t$:

1. Tenemos una capa densa con función de activación $\phi$ que toma nuestra matriz de diseño $\mathbf{X}_t$ y nuestra variable oculta $\mathbf{H}_{t-1}$.
2. A la salida generamos nuestra nueva variable oculta $\mathbf{H}_t$.
3. Con $\mathbf{H}_t$ y otra capa densa generamos nuestra salida $\mathbf{O}_t$

Muchas veces, en el paso 1 lo que se hace es concatenar $\mathbf{X}_t$ y $\mathbf{H}_{t-1}$ para de esa manera definir una unica matriz de pesos. A continuación mostramos como esta concatenación genera el mismo resultado.

In [1]:
import torch

In [2]:
X, W_xh = torch.randn(3, 1), torch.randn(1, 4)
H, W_hh = torch.randn(3, 4), torch.randn(4, 4)
torch.matmul(X, W_xh) + torch.matmul(H, W_hh)

tensor([[-0.0415,  0.0719,  0.9599, -2.4104],
        [-0.7444, -1.2778, -0.9349, -3.5133],
        [ 0.0190, -3.1796, -0.2329,  0.1189]])

In [3]:
torch.matmul(torch.cat((X, H), 1), torch.cat((W_xh, W_hh), 0))

tensor([[-0.0415,  0.0719,  0.9599, -2.4104],
        [-0.7444, -1.2778, -0.9349, -3.5133],
        [ 0.0190, -3.1796, -0.2329,  0.1189]])

# Preliminares a la implementación de RNN

Antes de discutir pasar a implementar una RNN desde cero, queremos discutir algunos temás más que serán importantes conocer.



## Muestro de secuencias.

Cuando tenemos que elegir que ejemplos debemos usar de un dataset que no contiene secuencias, simplemente mezclabamos aleatoriamente los ejemplos y luego los usabamos para entrenar nuestras redes.

Sin embargo, en secuencias temporales no podemos hacer esto. Si mezclamos aleatriamente podemos terminar generando secuencias sin sentido

> *En un lugar de la Mancha de cuyo nombre prefiero no acordarme*

luego de mezclarlo

> *nombre En de de no lugar prefiero la Mancha cuyo acordarme un*

Por esta razón debemos generar paritiones y mezclarlas. Por ejemplo podemos generar particiones de 4 elementos

> [*En un lugar de*] [*la Mancha de cuyo*] [*nombre prefiero no acordarme*]

> [*la Mancha de cuyo*][*nombre prefiero no acordarme*] [*En un lugar de*]

Además de, podemos elegir un offset o deplazamiento. En el ejemplo anterior, un offset de 1 generaría:

> *En* [*un lugar de la*] [*Mancha de cuyo nombre*] [*prefiero no acordarme, no*]



## Perplejidad

La perplejidad es una métrica que es usada en procesamiento de lenguajes naturales para tener una idea de que tan "convencido" está  nuestro modelo de la siguiente palabra que adivinará. Como métrica está relacionada a la entropía y la entropía cruzada.

Puede pensarse también a la perplejidad es una medida de la sorpresa o incertidumbre de un modelo de lenguaje al predecir la siguiente palabra en una secuencia. En otras palabras, indica qué tan bien un modelo de lenguaje puede capturar las dependencias estadísticas entre las palabras en un texto.

Cuanto menor sea la perplejidad, mejor será el modelo. Un modelo con baja perplejidad puede predecir con mayor precisión la siguiente palabra en una secuencia, lo que significa que tiene una mejor comprensión de las relaciones entre las palabras.

Técnicamente, la perplejidad no tiene un rango total definido, ya que puede tomar valores desde 0 hasta infinito.

• 0: Representaría un caso utópico en el que el modelo de lenguaje predice perfectamente la siguiente palabra en la secuencia, con un 100% de certeza.

• Infinito: Indicaría que el modelo no tiene idea de cuál podría ser la siguiente palabra, y asigna la misma probabilidad a todas las palabras posibles.

Sin embargo, en la práctica, es muy poco probable observar valores cercanos a estos extremos. En general, se considera que una perplejidad menor a 100 es buena, una perplejidad entre 100 y 300 es aceptable y una perplejidad superior a 300 es mala. Sin embargo, estos son solo valores de referencia y la interpretación de la perplejidad depende del contexto específico.

# Implementando una RNN desde 0

Ahora sí, implmentaremos una RNN desde 0



In [4]:
class RNNScratch(torch.nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.num_inputs = num_inputs
        self.W_xh = torch.nn.Parameter(
            torch.randn(num_inputs, num_hiddens) * 0.01)
        self.W_hh = torch.nn.Parameter(
            torch.randn(num_hiddens, num_hiddens) * 0.01)
        self.b_h = torch.nn.Parameter(torch.zeros(num_hiddens))

    def forward(self, inputs, state=None):
        if state is not None:
            state, = state
        outputs = []
        for X in inputs:  # Shape of inputs: (num_steps, batch_size, num_inputs)
            state = torch.tanh(torch.matmul(X, self.W_xh) + (
                torch.matmul(state, self.W_hh) if state is not None else 0)
                             + self.b_h)
            outputs.append(state)
        return outputs, state

Probemos que ocurre al generar una RNN y luego alimentarla con un tensor arbitrario

In [5]:
# 5 secuencias de longitud 3 de vectores de 4 componentes
A = torch.randn(5, 3, 4)
print(A[0]) # una secuencia de 3 vectores de 4 componentes

# red recurrente que toma vectores de 4 componentes y entrega vectores de 2
rnn =  RNNScratch(4, 2)

O, H = rnn(A)

print(len(O), O[0].shape) # 5 secuencias de longitud 3 de vectores de 2 componentes
print(H) # 3 vectores de 2 componentes

tensor([[-1.1987,  0.8580,  0.4982,  0.2459],
        [ 0.8903,  2.0783,  0.7507,  1.7596],
        [-0.7822,  0.4666,  1.5844,  0.0834]])
5 torch.Size([3, 2])
tensor([[-0.0020, -0.0144],
        [-0.0214, -0.0307],
        [-0.0135, -0.0072]], grad_fn=<TanhBackward0>)


Hay que considerar que hasta ahora solo hemos creado la variable oculta $\mathbf{H}$, no hemos aplicado la capa densa final que debemos usar para predecir la próxima palabra.

In [6]:
class RNNLMScratch(torch.nn.Module):
    """Defined in :numref:`sec_rnn-scratch`"""
    def __init__(self, rnn, vocab_size):
        super().__init__()
        self.rnn = rnn
        self.vocab_size = vocab_size
        self.init_params()

    def init_params(self):
        self.W_hq = torch.nn.Parameter(
            torch.randn(
                self.rnn.num_hiddens, self.vocab_size) * 0.01)
        self.b_q = torch.nn.Parameter(torch.zeros(self.vocab_size))

    def one_hot(self, X):
        """Defined in :numref:`sec_rnn-scratch`"""
        # Output shape: (num_steps, batch_size, vocab_size)
        return  torch.nn.functional.one_hot(X.T,
                                            self.vocab_size).type(torch.float32)

    def output_layer(self, rnn_outputs):
        """Defined in :numref:`sec_rnn-scratch`"""
        outputs = [torch.matmul(H, self.W_hq) + self.b_q for H in rnn_outputs]
        return torch.stack(outputs, 1)


    def forward(self, X, state=None):
        """Defined in :numref:`sec_rnn-scratch`"""
        embs = self.one_hot(X)
        rnn_outputs, _ = self.rnn(embs, state)
        return self.output_layer(rnn_outputs)

    def predict(self, prefix, num_preds, vocab, device=None):
        """Defined in :numref:`sec_rnn-scratch`"""
        state, outputs = None, [vocab[prefix[0]]]
        for i in range(len(prefix) + num_preds - 1):
            #alimentamos a la red para generar los estados ocultos a partir de los prefijos
            X = torch.tensor([[outputs[-1]]], device=device)
            embs = self.one_hot(X)
            rnn_outputs, state = self.rnn(embs, state)
            #if type(state) is tuple: state = state[0]
            if i < len(prefix) - 1:  # Warm-up period
                #pasamos la ground truth
                outputs.append(vocab[prefix[i + 1]])
            else:  # Predict `num_preds` steps
                #predicciones de la siguiente letra
                Y = self.output_layer(rnn_outputs)
                outputs.append(int(torch.reshape(torch.argmax(Y, axis=2),
                                                 (1,))))
        return ''.join([vocab.get_itos()[i] for i in outputs]) #el metodo .get_itos() realiza la conversion de numeros a letras

También aquí podemos revisar con que entramos a nuestra red y con que salimos.

In [7]:
model = RNNLMScratch(rnn, 4)
outputs = model(torch.ones((3, 5), dtype=torch.int64))
outputs.shape

torch.Size([3, 5, 4])

## Problemas con los gradientes y *Gradient clipping*

Las RNNs son propensas a dos problemas relacionados con los gradientes durante el entrenamiento, conocidos como desvanecimiento del gradiente y explosión del gradiente.

1. Desvanecimiento del Gradiente:

Este problema es el más común en las RNNs. Durante el backpropagation, los gradientes se propagan a través de los pasos de tiempo en la red. Si las funciones de activación utilizadas en la RNN tienen gradientes menores que 1 (en valor absoluto), estos gradientes pueden volverse muy pequeños a medida que se multiplican a través de múltiples pasos de tiempo. Esto dificulta que la red aprenda dependencias a largo plazo dentro de las secuencias, ya que las señales de error para los pasos de tiempo anteriores se vuelven insignificantes.

2. Explosión del Gradiente:

Este problema es el opuesto al desvanecimiento del gradiente. Si las funciones de activación tienen gradientes mayores que 1 (en valor absoluto), los gradientes pueden explotar exponencialmente durante la propagación hacia atrás, lo que lleva a un entrenamiento inestable y potencialmente hace que los pesos de la red se vuelvan muy grandes. Esto puede impedir que la red converja a una buena solución.


Para evitar un crecimiento descontrolado de nuestro gradiente, se realiza lo que se conoce como *gradient clipping*. Es decir, cuando el gradiente es mayor a cierta cantidad se procede a entregar un valor fijo de gradiente por encima del umbral definido. En efecto, lo que se hace es calcular el gradiente de la siguiente manera:

$$\mathbf{g} \leftarrow \min\left(1, \frac{\theta}{\|\mathbf{g}\|}\right) \mathbf{g}.$$

* $\mathbf{g}$: es el gradiente original.
* $θ$: es el valor umbral predefinido.
* $||g||$: es la norma del gradiente.
* $min(1, θ/||g||)$: es el valor que se utiliza para reemplazar el gradiente normalizado si este es mayor que el valor umbral.

De esta manera el modulo del gradiente nunca supera la cantidad $\theta$

In [8]:
def clip_gradients(grad_clip_val, model):
        params = [p for p in model.parameters() if p.requires_grad]
        norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
        if norm > grad_clip_val:
            for param in params:
                param.grad[:] *= grad_clip_val / norm

# Preprocesamiento

Recordemos un momento como es nuestro pipeline:

1. Carga de los datos
1. Separación de los datos en lotes
1. Inicialización de parámetros
1. Definición del modelo
1. Definición de la función de pérdida
1. Definición del algoritmo de optimización

En líneas generales hemos presentado todos los pasos de nuestro pipeline, pero debemos tener cuidado con el proceso de tokenización, como se explicó anteriormente. Por eso presentaremos algunas herramientas muy sencillas de tokenización.

Nuestra tarea, en este caso, sera tratar de enseñarle a nuestra red a escribir correctamente en español letra por letra. Para esto hemos elegido "El ingenioso hidalgo Don Quijote de la Mancha" como texto de referencia. Usaramos la letras del mismo texto para enseñarle a nuestro modelo a escribir palabras en español. Para eso tokenizaremos las letras del español.

In [9]:
import re
import collections
import torchtext
from torchtext.vocab import build_vocab_from_iterator


def make_vocab(fn, skip=0):
  data = None
  with open(fn, "r") as f:
    f.seek(skip)
    data = f.read()
  if data == None:
    return None, None
    # ".." match " "
    # ".;" match " "
    # ".a" match " a"
    # " . " match "." reemplaza "   "
    # " .. " match ".." reemplaza "   "
    # guía no machea nada "guía"
  tokens = re.sub('[^A-Za-záéíóúÁÉÍÓÚñÑüÜ]+', ' ', data).lower()
  tokens = [token for line in list(tokens) for token in line]
  counter = collections.Counter(tokens)
  freq_tuples = sorted(counter.items(), key=lambda x: x[1], reverse=True)
  ordered_dict = collections.OrderedDict(freq_tuples)
  result = build_vocab_from_iterator(ordered_dict)
  corpus = [result[token] for token in tokens]
  return result, ordered_dict, corpus

!wget https://www.gutenberg.org/files/2000/2000-0.txt
vocab, dictionary, corpus = make_vocab("2000-0.txt", 41508)

/usr/local/lib/python3.10/dist-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.10/dist-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


--2024-06-11 22:54:39--  https://www.gutenberg.org/files/2000/2000-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2226045 (2.1M) [text/plain]
Saving to: ‘2000-0.txt’

2000-0.txt          100%[===================>]   2.12M  --.-KB/s    in 0.1s    

2024-06-11 22:54:40 (14.2 MB/s) - ‘2000-0.txt’ saved [2226045/2226045]



In [43]:
type(vocab)

torchtext.vocab.vocab.Vocab

In [42]:
dictionary

OrderedDict([(' ', 379641),
             ('e', 221087),
             ('a', 192154),
             ('o', 152818),
             ('s', 125012),
             ('n', 108130),
             ('r', 100831),
             ('l', 88482),
             ('d', 86733),
             ('u', 77780),
             ('i', 77490),
             ('t', 62397),
             ('c', 59258),
             ('m', 44449),
             ('p', 35440),
             ('q', 32168),
             ('y', 25179),
             ('b', 24135),
             ('h', 20215),
             ('v', 17745),
             ('g', 17386),
             ('í', 12367),
             ('j', 10507),
             ('ó', 9069),
             ('f', 7810),
             ('é', 7110),
             ('á', 7036),
             ('z', 6430),
             ('ñ', 4210),
             ('ú', 1259),
             ('x', 399),
             ('w', 284),
             ('k', 133),
             ('ü', 84)])

## Generación de conjuntos de train y test

In [10]:
batch_size = 1024 #secuencias
num_steps = 32 #caracteres
array = torch.tensor([corpus[i:i+num_steps+1] #nos corremos de a 1 caracter
                            for i in range(0, len(corpus)-num_steps-1)])
# qwert y (i = 0)
# q werty (i = 1)
features, tags = array[:,:-1], array[:,1:]

num_train = 20480
num_val = 5120
def get_tensorloader(tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, batch_size,
                                        shuffle=train)

train_iter = get_tensorloader([features, tags], True, indices=slice(0, num_train))
test_iter = get_tensorloader([features, tags], False,
                             indices=slice(num_train, num_train + num_val))


In [11]:
print(vocab[" "])
print(vocab["e"])
print(vocab["a"])
print(vocab["á"])
print(vocab["ñ"])
print(type(vocab.get_itos()) is list)
print(vocab.get_itos())
print(vocab.get_itos()[9])

0
5
1
27
30
True
[' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'á', 'é', 'í', 'ñ', 'ó', 'ú', 'ü']
i


Haremos unas pequeña redifinición a nuestra función de pérdida, dado que estamos tamos trabajando con tensores con 3 dimensiones

In [12]:
def loss_NLP(Y_hat, Y):
    Y_hat = torch.reshape(Y_hat, (-1, Y_hat.shape[-1]))
    Y = torch.reshape(Y, (-1,))
    return torch.nn.functional.cross_entropy(
        Y_hat, Y, reduction='none')

# Entrenamiento

In [13]:
# Definimos el número de entradas (tamaño del vocabulario) y el número de neuronas en la capa oculta
rnn_scrt =  RNNScratch(num_inputs=len(vocab), num_hiddens=32)
net1 = RNNLMScratch(rnn_scrt, vocab_size=len(vocab))

# Definimos la función de pérdida y el optimizador
loss = loss_NLP
trainer = torch.optim.Adadelta(net1.parameters(), lr=1)

num_epochs = 100
# Iniciamos el bucle de entrenamiento
for epoch in range(num_epochs):
    L = 0.0 # Inicializamos la pérdida acumulada para la época actual
    N = 0 # Inicializamos el número total de elementos procesados en el conjunto de entrenamiento
    TestN = 0  # Inicializamos el número total de elementos procesados en el conjunto de evaluacion
    TestL = 0  # Inicializamos la pérdida acumulada para el conjunto de evaluacion

    # Iteramos sobre cada lote de datos en el conjunto de entrenamiento
    for X, Y in train_iter:
        # Calculamos la pérdida del modelo en el lote actual
        l = loss(net1(X), Y)
        trainer.zero_grad() # Reiniciamos los gradientes a cero
        l.mean().backward() # Realizamos la retropropagación para calcular los gradientes
        clip_gradients(grad_clip_val = 1, model = net1)  # Aplicamos gradient clipping para evitar gradientes explosivos
        trainer.step()  # Actualizamos los parámetros del modelo utilizando los gradientes calculados
        L += l.sum() # Acumulamos la pérdida
        N += l.numel()  # Acumulamos el número de elementos

    # Iteramos sobre cada lote de datos en el conjunto de evaluacion para evaluar el modelo
    for X, Y in test_iter:
        TestL += l.sum()
        TestN += Y.numel()

    # Imprimimos los resultados de la época actual
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')
    print(f'    train perplexity {torch.exp((L/N)):f},')
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')
    print()




epoch 1
    loss 3.479892,
    train perplexity 32.456203,
    test perplexity 29.297047.

epoch 2
    loss 2.946499,
    train perplexity 19.039179,
    test perplexity 17.387018.

epoch 3
    loss 2.848659,
    train perplexity 17.264616,
    test perplexity 17.096697.

epoch 4
    loss 2.839743,
    train perplexity 17.111366,
    test perplexity 17.154600.

epoch 5
    loss 2.819886,
    train perplexity 16.774933,
    test perplexity 16.391829.

epoch 6
    loss 2.762661,
    train perplexity 15.841949,
    test perplexity 15.145126.

epoch 7
    loss 2.655980,
    train perplexity 14.238935,
    test perplexity 13.469985.

epoch 8
    loss 2.561428,
    train perplexity 12.954297,
    test perplexity 12.605256.

epoch 9
    loss 2.492260,
    train perplexity 12.088565,
    test perplexity 11.787123.

epoch 10
    loss 2.438191,
    train perplexity 11.452304,
    test perplexity 11.254841.

epoch 11
    loss 2.396067,
    train perplexity 10.979906,
    test perplexity 10.806334

Veamos el resultado final.

In [14]:
net1.predict("quijote y sancho ", 40, vocab)

'quijote y sancho de la manta a la muestra de la manta a l'

# Implementación concisa de RNN

En función a los dos problemas asociados a Backpropagation en el tiempo y al crecieminto descontrolado de los gradientes, vemos que es preferible usar las herramientas que ya trae `torch`. Para llamar a una RNN que entrega estados ocultos deberemos hacer lo que se muestra en el siguiente código

In [15]:
class RNN(torch.nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()
        self.rnn = torch.nn.RNN(num_inputs, num_hiddens)

    def forward(self, inputs, H=None):
        return self.rnn(inputs, H)

In [16]:
class RNNLM(RNNLMScratch):
    def init_params(self):
        self.linear = torch.nn.LazyLinear(self.vocab_size)
    def output_layer(self, hiddens):
        return self.linear(hiddens).swapaxes(0, 1)

In [17]:
rnn =  RNN(num_inputs=len(vocab), num_hiddens=32)
net2 = RNNLM(rnn, vocab_size=len(vocab))
loss = loss_NLP
trainer = torch.optim.Adadelta(net2.parameters(), lr=1)

num_epochs = 100
for epoch in range(num_epochs):
    L = 0.0
    N = 0
    TestN = 0
    TestL = 0
    for X, Y in train_iter:
        l = loss(net2(X), Y)
        trainer.zero_grad()
        l.mean().backward()
        clip_gradients(grad_clip_val = 1, model = net2)
        trainer.step()
        L += l.sum()
        N += l.numel()
    for X, Y in test_iter:
        TestL += l.sum()
        TestN += Y.numel()
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')
    print(f'    train perplexity {torch.exp((L/N)):f},')
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')
    print()




/usr/local/lib/python3.10/dist-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


epoch 1
    loss 3.126075,
    train perplexity 22.784376,
    test perplexity 17.434460.

epoch 2
    loss 2.823086,
    train perplexity 16.828701,
    test perplexity 16.354778.

epoch 3
    loss 2.748584,
    train perplexity 15.620502,
    test perplexity 14.779106.

epoch 4
    loss 2.619095,
    train perplexity 13.723296,
    test perplexity 12.877835.

epoch 5
    loss 2.503941,
    train perplexity 12.230597,
    test perplexity 11.738267.

epoch 6
    loss 2.420964,
    train perplexity 11.256702,
    test perplexity 10.826333.

epoch 7
    loss 2.363052,
    train perplexity 10.623328,
    test perplexity 10.340177.

epoch 8
    loss 2.316016,
    train perplexity 10.135215,
    test perplexity 10.051311.

epoch 9
    loss 2.277884,
    train perplexity 9.756015,
    test perplexity 9.571691.

epoch 10
    loss 2.246031,
    train perplexity 9.450154,
    test perplexity 9.306638.

epoch 11
    loss 2.217340,
    train perplexity 9.182872,
    test perplexity 9.177752.

epo

In [44]:
net2.predict("quijote y sancho ", 40, vocab)

'quijote y sancho de como de la mando de la mando de la ma'

# Long Short-Term Memory (LSTM)

Uno de los problemas que vimos que tenían las redes recurrentes es las características de sus gradientes hacían que estos pudieran, o bien crecer de manera descontrolada, o bien achicarse hasta 0. En cualquiera de los dos casos nuestra propuesta de solución fue restringir el número de pasos hacia atras en el tiempo en los que calcularemos el gradiente. Sin embargo, este camino puede ser un problema en algunas aplicaciones.

Para esto surgió una alternativa a una RNN, que es la aquitectura LSTM

## Celdas de Memoria

LSTM es una red recurrente: la nueva salida depende de las entradas anteriores. Sin embargo, agrega un conjunto de variables ocultas para intentar emular la memoria RAM de una PC.

Una memoria RAM guarda diferente información para utilizarla luego en un cálculo. Además puede realizar un conjunto de operaciones por ejemplo:

* Leer los valores guardados anteriormente
* Escribir el valor guardado por uno nuevo
* Borrar lo que había en memoria.

La idea de la LSTM es trabajar con eso, pero hacerlo continuo. En este sentido LSTM tendrá dos variables ocultas. La primera es nuestra varaible oculta convencional $\mathbf{H}_{t-1}$. Pero la segunda es una varaible $\mathbf{C}_{t-1}$ que guarda información como una memoria RAM. Luego usaremos esa información para generar una nueva variable $\mathbf{H}_{t}$ en el próximo paso temporal.

En sintonía con lo anterior necesitaremos señales lógicas que nos ayudaran a decidir que hacer con la nueva entrada $\mathbf{X}_t$ y la varaible oculta anterior $\mathbf{H}_{t-1}$:

* Leer el valor de memoria $\mathbf{C}_{t-1}$ para calcular $\mathbf{H}_{t}$
* Modificar el valor anterior de $\mathbf{C}_{t-1}$, para crear uno nuevo $\mathbf{H}_t$
* Borrar completamente la memoria $\mathbf{C}_{t} = 0$$

En una memoria RAM real, estás señales son manejadas por señales binarias. Entonces si quisieramos borrar, pondríamos un 1 en entrada de la RAM que recibe la instrucción de borrado. Si quisieramos leer, pondríamos un 0 en la entrada de borrado y un 1 en la de leer, etc.

Al estar trabajando con tensores y problemas de optimización, ahora nuestra salida puede ser continua. Es decir, ya no solo podríamos borrar, sino tambien borrar parcialmente. Sin embargo, para esto debemos asegurarnos que nuestras salidas sean valores entre 0 y 1. Para esto crearemos una capa RNN con una sigmoidea como función de activación. Presentemos entonces las primeras 3 compuertas lógicas de nuestro LSTM:

### Compuestas Lógicas


![](http://d2l.ai/_images/lstm-0.svg)

$$
\begin{aligned}
\mathbf{O}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xo} + \mathbf{H}_{t-1} \mathbf{W}_{ho} + \mathbf{b}_o),\\
\mathbf{I}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xi} + \mathbf{H}_{t-1} \mathbf{W}_{hi} + \mathbf{b}_i),\\
\mathbf{F}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xf} + \mathbf{H}_{t-1} \mathbf{W}_{hf} + \mathbf{b}_f)
\end{aligned}
$$

Es decir, tenemos 3 capas RNN, con sus respectivos pesos. En donde:

* $\mathbf{O}_{t}$ corresponde a la señal de lectura. Nos dice que tanta importacia debemos darle a los valores anteriores de nuestra variable oculta $\mathbf{H}_{t-1}$
* $\mathbf{I}_{t}$ corresponde a la señal de escritura.Nos dice que tanto debe cambiar nuestra variable oculta anterior $\mathbf{H}_{t-1}$.
* $\mathbf{F}_{t}$ corresponde a la señal de borrado.Nos dice que tanto eliminar nuestra variable oculta anterior $\mathbf{H}_{t-1}$.

La analogía con el lenguaje es más o menos directa:

* En un libro de botánica, al describir un árbol hay una serie de oraciones relacionadas entre ellas. Nuestra red aprender a mantener la coherencia. Si hablamos de la fruta roja del árbol, no puede luego hablar de la fruta amarilla. Debemos MANTENER el tema de la conversación.
* En una obra de teatro, un personaje puede cambiar su estado. Puede pasar de estar parado a desmayarse. En ese sentido debemos poder MODIFICAR la situación del personajes. De otra manera, sería dificil de entender poque alguien se levanto si nunca dejo de estar parado.
* En una novela, se suceden una serie de acciones, pero no siempre están conectadas entre ellas. Si un capitulo esta centrado en el protagonista y el sigueinte centrado en el villano, debemos OLVIDAR el contexto anterior para no perder el hilo. Si antes el protagonista usaba patalones azules, debemos ignorar eso cuando los pantalones del villano son negros.

Rercordemos una vez más que hemos elegido una función de activación sigmoidea en analogía a las señales de las compuertas lógicas de una memoria RAM.

### Candidato de memoria

Ahora, lo que haremos será calcular el nuevo valor que guardaremos en memoria. Para esto simplementa aplicamos una RNN convencional con una $\tanh$ como función de activación.

$$\tilde{\mathbf{C}}_t = \text{tanh}(\mathbf{X}_t \mathbf{W}_{xc} + \mathbf{H}_{t-1} \mathbf{W}_{hc} + \mathbf{b}_c),$$

Este valor es un valor tentativo con el cual cambiaremos el valor existente en nuestra memoria $\mathbf{C}_t$

![](http://d2l.ai/_images/lstm-1.svg)

### Escribiendo en memoria.

Ahora lo que haremos será modificar nuestro valor en memoria. Hay dos operaciones que pueden modificar nuestra memoria: borrado y escritura. Con lo cual haremos una combinación lineal de las dos cosas

$$\mathbf{C}_t = \mathbf{F}_t \odot \mathbf{C}_{t-1} + \mathbf{I}_t \odot \tilde{\mathbf{C}}_t.$$

En donde hemos usado $\odot$ para representar el **temido** producto de Haddamar. Analicemos con un ejemplo sencillo como calcular el producto de Haddamar de dos matrices:


$$
\mathbf{A} = \begin{bmatrix}2&3\\5&7\end{bmatrix},
\mathbf{B} = \begin{bmatrix}3&5\\7&2\end{bmatrix},\\
\mathbf{A} \odot \mathbf{B} = \begin{bmatrix}2&3\\5&7\end{bmatrix} ⊙ \begin{bmatrix}3&5\\7&2\end{bmatrix}=\begin{bmatrix}6&15\\35&14\end{bmatrix}\\
\mathbf{A} \odot \mathbf{A} = \begin{bmatrix}2&3\\5&7\end{bmatrix} ⊙ \begin{bmatrix}2&3\\5&7\end{bmatrix}=\begin{bmatrix}4&9\\25&49\end{bmatrix}
$$

Vemos que el producto de Haddamar es en esencia no es más que multiplicar elemento a elemento de una matriz o un tensor. Es simplemente un nombre raro, para algo que es mucho más intuitivo que la multiplicación de matrices tradicionales.

![](http://d2l.ai/_images/lstm-2.svg)

Conviene analizar que ocurre en cada caso para $\mathbf{F}_{t}$ y $\mathbf{I}_{t}$

||$\mathbf{I}_{t} = 1$| $\mathbf{I}_{t} = 0$
|---|:-:|:-:|
|$\mathbf{F}_{t}=1$|Combinación lineal del valor nuevo y el viejo|Se conserva el valor viejo|
|$\mathbf{F}_{t}=0$|Se reemplaza el valor nuevo por el viejo|Se borra la celda de memoria|

### Estado oculto.

Ahora sí, leeremos la memoria para obtener nuestro nueva variable oculta

$$\mathbf{H}_t = \mathbf{O}_t \odot \tanh(\mathbf{C}_t).$$

Es decir, aplicamos una última transformación a nuestro valor de memoria y luego decidimos cuanto leeremos de ese valor. Si $\mathbf{O}_t = 0$, ignoraremos lo que haya en memoria, pero si $\mathbf{O}_t = 1$, le prestaremos total antención.

![](http://d2l.ai/_images/lstm-3.svg)


## Implementation de LSTM desde 0


In [19]:
class LSTMScratch(torch.nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()

        init_weight = lambda *shape: torch.nn.Parameter(torch.randn(*shape) * 0.01)
        triple = lambda: (init_weight(num_inputs, num_hiddens),
                          init_weight(num_hiddens, num_hiddens),
                          torch.nn.Parameter(torch.zeros(num_hiddens)))
        self.num_hiddens = num_hiddens
        self.num_inputs = num_inputs
        self.W_xi, self.W_hi, self.b_i = triple()  # Input gate
        self.W_xf, self.W_hf, self.b_f = triple()  # Forget gate
        self.W_xo, self.W_ho, self.b_o = triple()  # Output gate
        self.W_xc, self.W_hc, self.b_c = triple()  # Candidate memory cell

    def forward(self, inputs, H_C=None):
        H, C = None, None if H_C is None else H_C
        outputs = []
        for X in inputs:
            I = torch.sigmoid(torch.matmul(X, self.W_xi) + (
                torch.matmul(H, self.W_hi) if H is not None else 0) + self.b_i)
            if H is None:
                H, C = torch.zeros_like(I), torch.zeros_like(I)
            F = torch.sigmoid(torch.matmul(X, self.W_xf) +
                            torch.matmul(H, self.W_hf) + self.b_f)
            O = torch.sigmoid(torch.matmul(X, self.W_xo) +
                            torch.matmul(H, self.W_ho) + self.b_o)
            C_tilda = torch.tanh(torch.matmul(X, self.W_xc) +
                              torch.matmul(H, self.W_hc) + self.b_c)
            C = F * C + I * C_tilda # El prod. de Haddar es el producto normal!
            H = O * torch.tanh(C)
            outputs.append(H)
        return outputs, (H, C)

### Entrenamiento




In [20]:
lstm_scrt = LSTMScratch(num_inputs=len(vocab), num_hiddens=32)
net3 = RNNLMScratch(lstm_scrt, vocab_size=len(vocab))
trainer = torch.optim.Adadelta(net3.parameters(), lr=4)
loss = loss_NLP

num_epochs = 100
for epoch in range(num_epochs):
    L = 0.0
    N = 0
    TestN = 0
    TestL = 0
    for X, Y in train_iter:
        l = loss(net3(X), Y)
        trainer.zero_grad()
        l.mean().backward()
        clip_gradients(grad_clip_val = 1, model = net3)
        trainer.step()
        L += l.sum()
        N += l.numel()
    for X, Y in test_iter:
        TestL += l.sum()
        TestN += Y.numel()
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')
    print(f'    train perplexity {torch.exp((L/N)):f},')
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')
    print()


epoch 1
    loss 3.274318,
    train perplexity 26.425203,
    test perplexity 17.836447.

epoch 2
    loss 2.855389,
    train perplexity 17.381191,
    test perplexity 17.248619.

epoch 3
    loss 2.838872,
    train perplexity 17.096478,
    test perplexity 17.035973.

epoch 4
    loss 2.823643,
    train perplexity 16.838087,
    test perplexity 16.762625.

epoch 5
    loss 2.799876,
    train perplexity 16.442612,
    test perplexity 16.140945.

epoch 6
    loss 2.765229,
    train perplexity 15.882684,
    test perplexity 15.513397.

epoch 7
    loss 2.703747,
    train perplexity 14.935584,
    test perplexity 14.372225.

epoch 8
    loss 2.598460,
    train perplexity 13.443019,
    test perplexity 12.708462.

epoch 9
    loss 2.482206,
    train perplexity 11.967640,
    test perplexity 11.479474.

epoch 10
    loss 2.401349,
    train perplexity 11.038054,
    test perplexity 10.857491.

epoch 11
    loss 2.353207,
    train perplexity 10.519255,
    test perplexity 10.213072

In [21]:
net3.predict("sancho y quijote", 30, vocab)

'sancho y quijote a a a a a a a a a a a a a a a'

## Implementación Concisa


In [22]:
class LSTM(RNN):
    def __init__(self, num_inputs, num_hiddens):
        torch.nn.Module.__init__(self)
        self.rnn = torch.nn.LSTM(num_inputs, num_hiddens)

    def forward(self, inputs, H_C=None):
        return self.rnn(inputs, H_C)

In [23]:
lstm = LSTM(num_inputs=len(vocab), num_hiddens=32)
net4 = RNNLM(lstm, vocab_size=len(vocab))
trainer = torch.optim.Adadelta(net4.parameters(), lr=4)
loss = loss_NLP

num_epochs = 100
for epoch in range(num_epochs):
    L = 0.0
    N = 0
    TestN = 0
    TestL = 0
    for X, Y in train_iter:
        l = loss(net4(X), Y)
        trainer.zero_grad()
        l.mean().backward()
        clip_gradients(grad_clip_val = 1, model = net4)
        trainer.step()
        L += l.sum()
        N += l.numel()
    for X, Y in test_iter:
        TestL += l.sum()
        TestN += Y.numel()
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')
    print(f'    train perplexity {torch.exp((L/N)):f},')
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')
    print()


epoch 1
    loss 3.056890,
    train perplexity 21.261332,
    test perplexity 17.324142.

epoch 2
    loss 2.826025,
    train perplexity 16.878229,
    test perplexity 16.439095.

epoch 3
    loss 2.752555,
    train perplexity 15.682655,
    test perplexity 14.628424.

epoch 4
    loss 2.577667,
    train perplexity 13.166385,
    test perplexity 11.923112.

epoch 5
    loss 2.417523,
    train perplexity 11.218034,
    test perplexity 10.448276.

epoch 6
    loss 2.327736,
    train perplexity 10.254703,
    test perplexity 9.953538.

epoch 7
    loss 2.259852,
    train perplexity 9.581671,
    test perplexity 9.456846.

epoch 8
    loss 2.214936,
    train perplexity 9.160821,
    test perplexity 9.010310.

epoch 9
    loss 2.173868,
    train perplexity 8.792224,
    test perplexity 8.641846.

epoch 10
    loss 2.145594,
    train perplexity 8.547113,
    test perplexity 8.486083.

epoch 11
    loss 2.112514,
    train perplexity 8.269000,
    test perplexity 8.227973.

epoch 12

In [24]:
net4.predict('levantose ', 20, vocab)

'levantose en el caballero a su'

# Gated Recurrent Units (GRU)

La arquitectura de LSTM es una arquitectura de la década de 1990. Sin embargo, en 2014, de desarrolló una alternativa más simple que LSTM y que tiene un comportamiento similar. Esta es GRU.

### Compuertas lógicas

Al igual que LSTM, GRU tambien usa unas compuertas lógicas con salidas entre 0 y 1

![Computing the reset gate and the update gate in a GRU model.](https://d2l.ai/_images/gru-1.svg)


$$
\begin{aligned}
\mathbf{R}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xr} + \mathbf{H}_{t-1} \mathbf{W}_{hr} + \mathbf{b}_r),\\
\mathbf{Z}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xz} + \mathbf{H}_{t-1} \mathbf{W}_{hz} + \mathbf{b}_z),
\end{aligned}
$$

### Candidato de variable oculta

Luego, en lugar de calcular el valor de una celda de memoria, GRU directamente propone un nuevo valor de variable oculta.

$$\tilde{\mathbf{H}}_t = \tanh(\mathbf{X}_t \mathbf{W}_{xh} + \left(\mathbf{R}_t \odot \mathbf{H}_{t-1}\right) \mathbf{W}_{hh} + \mathbf{b}_h),$$

En donde, si $\mathbf{R}_t = 0$ ignoramos o *reseteamos* la varaible oculta. Pero si $\mathbf{R}_t =1$ conservamos toda su información.

![](https://d2l.ai/_images/gru-2.svg)

Destacamos que hemos vuelto a usar el producto de Haddamar o producto elemento a elemento.

### Variable oculta

Finalmente usamos nuestra otra compurta lógica para definir que tanta importación le damos al candidato nuevo actual con respecto al valor anteior anterior.

$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1}  + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t.$$

Es decir, si $\mathbf{Z}_t = 1$  conservamos el valor anteior y no actualizamos nuestra variable. Pero si $\mathbf{Z}_t = 0$, ignoramos el valor anterior y conservamos al candidato.

![](https://d2l.ai/_images/gru-3.svg)





## Implementación de GRU desde 0

In [25]:
class GRUScratch(torch.nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()

        init_weight = lambda *shape: torch.nn.Parameter(torch.randn(*shape) * 0.01)
        triple = lambda: (init_weight(num_inputs, num_hiddens),
                          init_weight(num_hiddens, num_hiddens),
                          torch.nn.Parameter(torch.zeros(num_hiddens)))
        self.num_hiddens = num_hiddens
        self.num_inputs = num_inputs
        self.W_xz, self.W_hz, self.b_z = triple()  # Update gate
        self.W_xr, self.W_hr, self.b_r = triple()  # Reset gate
        self.W_xh, self.W_hh, self.b_h = triple()  # Candidate hidden state

    def forward(self, inputs, H=None):
        matmul_H = lambda A, B: torch.matmul(A, B) if H is not None else 0
        outputs = []
        for X in inputs:
            Z = torch.sigmoid(torch.matmul(X, self.W_xz) + (
                torch.matmul(H, self.W_hz) if H is not None else 0) + self.b_z)
            if H is None: H = torch.zeros_like(Z)
            R = torch.sigmoid(torch.matmul(X, self.W_xr) +
                            torch.matmul(H, self.W_hr) + self.b_r)
            # R * H es otro producto de Haddamar!!
            H_tilda = torch.tanh(torch.matmul(X, self.W_xh) +
                              torch.matmul(R * H, self.W_hh) + self.b_h)
            H = Z * H + (1 - Z) * H_tilda # Mas prod. de Haddamar!
            outputs.append(H)
        return outputs, H

### Entrenamiento


In [26]:
gru_scrt = GRUScratch(num_inputs=len(vocab), num_hiddens=32)
net5 = RNNLMScratch(gru_scrt, vocab_size=len(vocab))
trainer = torch.optim.Adadelta(net5.parameters(), lr=4)
loss = loss_NLP

num_epochs = 100
for epoch in range(num_epochs):
    L = 0.0
    N = 0
    TestN = 0
    TestL = 0
    for X, Y in train_iter:
        l = loss(net5(X), Y)
        trainer.zero_grad()
        l.mean().backward()
        clip_gradients(grad_clip_val = 1, model = net5)
        trainer.step()
        L += l.sum()
        N += l.numel()
    for X, Y in test_iter:
        TestL += l.sum()
        TestN += Y.numel()
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')
    print(f'    train perplexity {torch.exp((L/N)):f},')
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')
    print()

epoch 1
    loss 3.151219,
    train perplexity 23.364532,
    test perplexity 17.414799.

epoch 2
    loss 2.842757,
    train perplexity 17.163015,
    test perplexity 16.801815.

epoch 3
    loss 2.759023,
    train perplexity 15.784410,
    test perplexity 14.585828.

epoch 4
    loss 2.566084,
    train perplexity 13.014764,
    test perplexity 11.874869.

epoch 5
    loss 2.415889,
    train perplexity 11.199720,
    test perplexity 10.677231.

epoch 6
    loss 2.325484,
    train perplexity 10.231629,
    test perplexity 9.821965.

epoch 7
    loss 2.263319,
    train perplexity 9.614953,
    test perplexity 9.359979.

epoch 8
    loss 2.216442,
    train perplexity 9.174626,
    test perplexity 8.951295.

epoch 9
    loss 2.177448,
    train perplexity 8.823762,
    test perplexity 8.684489.

epoch 10
    loss 2.145526,
    train perplexity 8.546535,
    test perplexity 8.468608.

epoch 11
    loss 2.116988,
    train perplexity 8.306086,
    test perplexity 8.129966.

epoch 12

In [27]:
net5.predict("levantose ", 30, vocab)

'levantose a la mande al caballero con es'

## Implementación concisa

In [28]:
class GRU(RNN):
    def __init__(self, num_inputs, num_hiddens):
        torch.nn.Module.__init__(self)
        self.rnn = torch.nn.GRU(num_inputs, num_hiddens)

In [29]:
gru = GRU(num_inputs=len(vocab), num_hiddens=32)
net6 = RNNLM(gru, vocab_size=len(vocab))
trainer = torch.optim.Adadelta(net6.parameters(), lr=4)
loss = loss_NLP

num_epochs = 100
for epoch in range(num_epochs):
    L = 0.0
    N = 0
    TestN = 0
    TestL = 0
    for X, Y in train_iter:
        l = loss(net6(X), Y)
        trainer.zero_grad()
        l.mean().backward()
        clip_gradients(grad_clip_val = 1, model = net6)
        trainer.step()
        L += l.sum()
        N += l.numel()
    for X, Y in test_iter:
        TestL += l.sum()
        TestN += Y.numel()
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')
    print(f'    train perplexity {torch.exp((L/N)):f},')
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')
    print()

epoch 1
    loss 3.008144,
    train perplexity 20.249779,
    test perplexity 16.949184.

epoch 2
    loss 2.708590,
    train perplexity 15.008100,
    test perplexity 12.802689.

epoch 3
    loss 2.408299,
    train perplexity 11.115041,
    test perplexity 10.194564.

epoch 4
    loss 2.271796,
    train perplexity 9.696798,
    test perplexity 9.352798.

epoch 5
    loss 2.199068,
    train perplexity 9.016606,
    test perplexity 8.744968.

epoch 6
    loss 2.148183,
    train perplexity 8.569273,
    test perplexity 8.283027.

epoch 7
    loss 2.106528,
    train perplexity 8.219653,
    test perplexity 8.089149.

epoch 8
    loss 2.068496,
    train perplexity 7.912917,
    test perplexity 7.695341.

epoch 9
    loss 2.034891,
    train perplexity 7.651419,
    test perplexity 7.531065.

epoch 10
    loss 2.007996,
    train perplexity 7.448377,
    test perplexity 7.337418.

epoch 11
    loss 1.981856,
    train perplexity 7.256199,
    test perplexity 7.195019.

epoch 12
    

In [30]:
net6.predict('avellaneda ', 20, vocab)

'avellaneda de la mendo a las ar'

# Redes bidireccionales y profundas

En general, vamos a ver que muchas veces tener una variable oculta lineal o generada por una sola capa, puede no capturar toda la complejidad de nuestro modelo. Es por esto que debemos tener alguna herramienta que nos permita generar un estado oculto mucho más complejo. Para esto tenemos las redes recurrentes profundas.

En líneas generales, la idea será usar sucesivos redes recurrentes a la salida de nuestra primera capa con estados ocultos. Como muestra la figura

![](https://d2l.ai/_images/deep-rnn.svg)

Afortunadamente este tipo de arquitecturas podemos llamarlas solo agregando un parametro a nuestro código.

In [31]:
class LSTMDeep(RNN):
    def __init__(self, num_inputs, num_hiddens,num_layers):
        torch.nn.Module.__init__(self)
        self.rnn = torch.nn.LSTM(num_inputs, num_hiddens,num_layers)
        self.num_hiddens = num_hiddens
        self.num_inputs = num_inputs
        self.num_layers = num_layers


    def forward(self, inputs, H_C=None):
        return self.rnn(inputs, H_C)

In [32]:
lstmD = LSTMDeep(num_inputs=len(vocab), num_hiddens=32, num_layers=2)
net7 = RNNLM(lstm, vocab_size=len(vocab))

Más adelante, veremos que en el problema de traducción, tener información del futuro puede resultar util para los estados presentes. El ejemplo más sencillo es como las palabras se ordenan de manera distinta segun el idioma:

>This is a red pencil

>Este un lápiz es rojo

Por esto también es útil tener los llamados modelos bidireccionales. Estos modelos duplican el número de parametros, pues en esencia entrenan tanto "hacia adelante" temporalmente como "hacia atras"

In [33]:
class LSTMBidir(RNN):
    def __init__(self, num_inputs, num_hiddens, num_layers, bidirectional):
        torch.nn.Module.__init__(self)
        self.rnn = torch.nn.LSTM(num_inputs,
                                 num_hiddens, num_layers,
                                 bidirectional=bidirectional)
        self.num_hiddens = num_hiddens
        if bidirectional: self.num_hiddens *= 2
        self.num_inputs = num_inputs
        self.num_layers = num_layers

    def forward(self, inputs, H_C=None):
        return self.rnn(inputs, H_C)

In [34]:
lstmD = LSTMBidir(num_inputs=len(vocab), num_hiddens=32,
                  num_layers=2, bidirectional=True)
net8 = RNNLM(lstm, vocab_size=len(vocab))

Cabe destacar que las redes bidireccionales son un mal modelo para el problema a "aprender a escribir español letra por letra". La razón, de alguna manera es la siguiente:

* Dada una letra "q", ¿cuál es la letra anterior en un texto?
* Dada una letra "q", ¿cuál es la letra siguiente en un texto?

A diferencia de la traducción automática, el deletro tiene mucha más información en la dirección "hacia adelante" que hacia atras.